<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/05-Statistics-and-Fitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/mjcowley/Python_Intro/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics Research Group" width="400"/>

# Part 5: Statistics and Fitting

**QUT Astrophysics Research Group (QARG) Python Training**

Group leads: Dr Michael Cowley and Dr Rossana Ruggeri
Author: Michael Cowley
Last updated: 18 Oct 2026

## Overview

Plots show you what's going on. Statistics tell you how sure you can be. This part covers the statistical tools you'll use most in a research project: summarising data robustly, propagating uncertainties, fitting models to data, and judging whether a difference is real.

We'll use two real datasets: the ZFOURGE catalogue from Parts 3 and 4, and the **Pantheon+** compilation of type Ia supernovae, which you'll use to measure the expansion rate of the Universe and find evidence for dark energy.

This notebook takes about **3 hours**. It's not a replacement for a statistics course, but it will give you working tools and point you to where to learn more.

### By the end of Part 5 you'll be able to

- Choose between the mean and median, and the standard deviation and robust alternatives, when your data has outliers
- Measure how good photometric redshifts are
- Propagate uncertainties, both with formulae and by simulation
- Combine measurements with a weighted mean, and quantify the tension between them
- Fit models to data with `scipy.optimize.curve_fit`, get uncertainties on the parameters, and judge the fit with $\chi^2$
- Compare two samples, and estimate uncertainties with the bootstrap

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

from astropy.cosmology import FlatLambdaCDM
from astropy.stats import sigma_clipped_stats
from astropy.table import Table
from astropy.utils.data import download_file

DATA_URL = "https://raw.githubusercontent.com/mjcowley/Python_Intro/main/data/"


def get_data(filename):
    """Download a course data file (only the first time) and return its local path."""
    return download_file(DATA_URL + filename, cache=True)

Load the ZFOURGE catalogue as in Part 4:

In [ ]:
df = pd.read_csv(get_data("zfourge_cdfs.csv"), na_values=[-99])
eazy = Table.read(get_data("zfourge_cdfs_eazy.fits"), hdu="EAZY").to_pandas().replace(-99, np.nan)
fast = pd.read_csv(get_data("zfourge_cdfs.fout"), sep=r"\s+", comment="#", na_values=[-99],
                   names=["id", "z", "ltau", "metal", "lage", "Av", "lmass", "lsfr", "lssfr", "la2t", "chi2"])

cat = pd.merge(df, eazy[["id", "z_peak", "l68", "u68", "f_U", "e_U", "f_V", "e_V", "f_J", "e_J"]], on="id")
cat = pd.merge(cat, fast[["id", "z", "lmass", "lsfr"]].rename(columns={"z": "z_best"}), on="id")

gal = cat[cat["use"] == 1].copy()
gal["Ks_mag"] = 25 - 2.5 * np.log10(gal["f_Ks"].where(gal["f_Ks"] > 0))
gal["u_v"] = -2.5 * np.log10(gal["f_U"] / gal["f_V"])
gal["v_j"] = -2.5 * np.log10(gal["f_V"] / gal["f_J"])
gal["quiescent"] = (gal["u_v"] > 1.3) & (gal["v_j"] < 1.6) & (gal["u_v"] > 0.88 * gal["v_j"] + 0.59)

print(f"{len(gal)} galaxies")

---
## 1. Robust Statistics

### When the mean misleads

The mean and standard deviation are the most familiar summaries of a dataset, but a single bad value can drag them a long way. Here are ten measurements of the same galaxy's redshift, one of which went badly wrong:

In [ ]:
z_measured = np.array([1.212, 1.208, 1.215, 1.211, 1.209, 1.214, 1.210, 1.213, 2.85, 1.211])

print(f"Mean:   {np.mean(z_measured):.3f}    Standard deviation: {np.std(z_measured, ddof=1):.3f}")
print(f"Median: {np.median(z_measured):.3f}")

The mean is pulled well away from the true value, and the standard deviation is dominated by the one outlier. The **median** barely notices it. Statistics that aren't thrown off by outliers are called **robust**.

The robust equivalent of the standard deviation is based on the **median absolute deviation** (MAD), the median distance of the points from their median. Multiplying it by 1.4826 gives the **normalised MAD** (NMAD), which equals the standard deviation for normally distributed data, so the two can be compared directly:

$$\sigma_\mathrm{NMAD} = 1.4826 \times \mathrm{median}\left(\left|x_i - \mathrm{median}(x)\right|\right)$$

In [ ]:
def nmad(x):
    """Normalised median absolute deviation: a robust estimate of the standard deviation."""
    x = np.asarray(x)
    return 1.4826 * np.median(np.abs(x - np.median(x)))


print(f"Standard deviation: {np.std(z_measured, ddof=1):.4f}")
print(f"NMAD:               {nmad(z_measured):.4f}")

Another common approach is **sigma clipping**: repeatedly throw away points more than (say) $3\sigma$ from the median, and recalculate. Astropy's `sigma_clipped_stats` does this and returns the clipped mean, median and standard deviation.

In [ ]:
mean, median, std = sigma_clipped_stats(z_measured, sigma=3)
print(f"Clipped mean {mean:.4f}, median {median:.4f}, std {std:.4f}")

Don't use clipping to quietly delete data you don't like, though. Outliers are often the most interesting objects in a sample, and you should always look at what you've removed and say how many in your write-up.

### How good are photometric redshifts?

Here's a real use of robust statistics. About 1,100 ZFOURGE galaxies have a spectroscopic redshift, which we treat as the truth. We compare each galaxy's photometric redshift to it using

$$\Delta z = \frac{z_\mathrm{phot} - z_\mathrm{spec}}{1 + z_\mathrm{spec}}$$

The factor of $1 + z$ allows for the fact that a given error in wavelength becomes a bigger error in redshift for more distant galaxies. Photo-z quality is normally reported with two numbers: the NMAD of $\Delta z$ (the typical scatter) and the **outlier fraction**, the fraction with $|\Delta z| > 0.15$.

In [ ]:
spec = gal[gal["z_spec"].notna()].copy()
spec["dz"] = (spec["z_peak"] - spec["z_spec"]) / (1 + spec["z_spec"])

print(f"{len(spec)} galaxies with spectroscopic redshifts")
print(f"Standard deviation of dz: {spec['dz'].std():.4f}")
print(f"NMAD of dz:               {nmad(spec['dz']):.4f}")
print(f"Median dz (bias):         {spec['dz'].median():+.4f}")
print(f"Outlier fraction:         {(spec['dz'].abs() > 0.15).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(spec["dz"], bins=np.linspace(-0.2, 0.2, 81), color="0.5")
ax.axvline(-0.15, color="k", ls="--")
ax.axvline(0.15, color="k", ls="--", label="Outlier threshold")
ax.set_xlabel(r"$\Delta z = (z_\mathrm{phot} - z_\mathrm{spec}) / (1 + z_\mathrm{spec})$")
ax.set_ylabel("Number of galaxies")
ax.legend()
plt.show()

The two scatter estimates differ by a factor of ten! The standard deviation of 0.12 is dominated by the few percent of catastrophic outliers you saw in Part 4. For the large majority of galaxies, the photometric redshift is good to about 1%, which is what [Straatman et al. (2016)](https://ui.adsabs.harvard.edu/abs/2016ApJ...830...51S) report for ZFOURGE. Both numbers matter: the NMAD tells you how precise a typical photo-z is, and the outlier fraction tells you how often it's badly wrong.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Where Do Photo-zs Struggle?
  </h1>
</div>

Using `spec` and your `nmad()` function:

1. Calculate the NMAD and outlier fraction for **bright** galaxies ($K_s < 22$) and **faint** galaxies ($K_s \geq 22$). Store the faint NMAD in `nmad_faint`.
2. Do the same for galaxies with $z_\mathrm{spec} < 1.5$ and $z_\mathrm{spec} \geq 1.5$. Store the high-redshift outlier fraction in `outlier_highz`.
3. Which galaxies have the least reliable photometric redshifts? Can you think of reasons why?
4. Spectroscopic samples aren't random (Part 4, Section 3). How might this affect what these numbers tell you about the photo-zs of the *whole* catalogue?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(nmad_faint - 0.0120) < 0.001, "nmad_faint isn't right yet. Use the galaxies with Ks >= 22."
assert abs(outlier_highz - 0.135) < 0.005, "outlier_highz isn't right yet. Use the fraction with |dz| > 0.15 at z_spec >= 1.5."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
bright = spec["Ks_mag"] < 22
low_z = spec["z_spec"] < 1.5

for label, mask in [("Ks < 22", bright), ("Ks >= 22", ~bright), ("z < 1.5", low_z), ("z >= 1.5", ~low_z)]:
    dz = spec.loc[mask, "dz"]
    print(f"{label:9s} N = {len(dz):4d}   NMAD = {nmad(dz):.4f}   outliers = {(dz.abs() > 0.15).mean():.1%}")

nmad_faint = nmad(spec.loc[~bright, "dz"])
outlier_highz = (spec.loc[~low_z, "dz"].abs() > 0.15).mean()

# Faint galaxies have noisier photometry, so their SEDs are less well constrained. At high redshift
# the key spectral features move into the near-infrared, where there are fewer filters, and the
# Lyman and Balmer breaks can be confused with each other. Spectroscopic targets are mostly bright,
# so the full catalogue, which is dominated by faint galaxies, will do worse than these numbers suggest.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 2. Propagating Uncertainties

Every measurement has an uncertainty. When you calculate something from measurements, the uncertainty carries through, or **propagates**, into the result.

It helps to separate two kinds of uncertainty:

- **Random** (statistical) uncertainties, like photon noise, scatter in both directions and shrink when you average many measurements.
- **Systematic** uncertainties, like a wrong calibration, shift everything the same way and don't average away. They're often the harder ones to deal with.

### The formula

For a quantity $f(x, y)$ calculated from independent measurements $x \pm \sigma_x$ and $y \pm \sigma_y$, the standard propagation formula is

$$\sigma_f^2 = \left(\frac{\partial f}{\partial x}\right)^2 \sigma_x^2 + \left(\frac{\partial f}{\partial y}\right)^2 \sigma_y^2$$

It assumes the uncertainties are small enough that $f$ is roughly linear over their range. A very useful case is converting a flux to a magnitude, $m = -2.5\log_{10} f$, which gives

$$\sigma_m = \frac{2.5}{\ln 10} \frac{\sigma_f}{f} \approx 1.086 \frac{\sigma_f}{f}$$

so a 10% flux uncertainty is roughly a 0.1 magnitude uncertainty.

In [ ]:
flux, flux_err = 12.0, 1.2          # 10% uncertainty

mag_err = 2.5 / np.log(10) * flux_err / flux
print(f"Magnitude uncertainty: {mag_err:.3f} mag")

### Monte Carlo propagation

The alternative is to **simulate**. Draw many random values of each measurement from its uncertainty distribution, calculate the result for each one, and look at the spread of results. It's slower, but it works for any calculation, however complicated, and it doesn't need the uncertainties to be small.

Here's the same flux-to-magnitude conversion, first for the 10% uncertainty above and then for a faint source with a 40% uncertainty:

In [ ]:
rng = np.random.default_rng(seed=42)

for flux, flux_err in [(12.0, 1.2), (3.0, 1.2)]:
    samples = rng.normal(flux, flux_err, size=100_000)
    samples = samples[samples > 0]                      # magnitudes need positive flux
    mags = 25 - 2.5 * np.log10(samples)

    p16, p50, p84 = np.percentile(mags, [16, 50, 84])
    formula = 2.5 / np.log(10) * flux_err / flux
    print(f"S/N = {flux / flux_err:.1f}: formula gives ±{formula:.3f}, "
          f"simulation gives -{p50 - p16:.3f} / +{p84 - p50:.3f}")

At S/N = 10 the two agree. At S/N = 2.5 the simulation shows the uncertainty is lopsided, because the logarithm stretches the faint side. The formula can't capture that, which is why magnitudes of faint sources often have asymmetric error bars.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Uncertainty on a Colour
  </h1>
</div>

A ZFOURGE galaxy has rest-frame fluxes $f_U = 0.2073 \pm 0.0131$ and $f_V = 0.3820 \pm 0.0160$.

1. Calculate its colour $U-V = -2.5\log_{10}(f_U / f_V)$.
2. Use the propagation formula to find the uncertainty on $U-V$. Each flux contributes like a magnitude, so $\sigma_{U-V} = 1.086\sqrt{(\sigma_U/f_U)^2 + (\sigma_V/f_V)^2}$. Store it in `uv_err_formula`.
3. Find the uncertainty by Monte Carlo instead, with 100,000 samples, and store it (the standard deviation of the simulated colours) in `uv_err_mc`. Do they agree?
4. **Extension:** The real catalogue has `e_U` and `e_V` columns. Add a column `u_v_err` to `gal` using the formula, and find the median uncertainty on $U-V$ for galaxies with $\log_{10}(M_*/M_\odot) > 10$. Is it small compared to the size of the UVJ diagram?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(uv_err_formula - 0.0822) < 0.001, "uv_err_formula isn't right yet. Check the formula."
assert abs(uv_err_mc - uv_err_formula) < 0.005, "Your Monte Carlo estimate should be close to the formula."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
f_u, e_u = 0.2073, 0.0131
f_v, e_v = 0.3820, 0.0160

u_v = -2.5 * np.log10(f_u / f_v)
uv_err_formula = 2.5 / np.log(10) * np.sqrt((e_u / f_u) ** 2 + (e_v / f_v) ** 2)

rng = np.random.default_rng(seed=1)
u_samples = rng.normal(f_u, e_u, size=100_000)
v_samples = rng.normal(f_v, e_v, size=100_000)
uv_samples = -2.5 * np.log10(u_samples / v_samples)
uv_err_mc = np.std(uv_samples)

print(f"U-V = {u_v:.3f} ± {uv_err_formula:.3f} (formula), ± {uv_err_mc:.3f} (Monte Carlo)")

gal["u_v_err"] = 2.5 / np.log(10) * np.sqrt((gal["e_U"] / gal["f_U"]) ** 2 + (gal["e_V"] / gal["f_V"]) ** 2)
print(f"Median U-V uncertainty for log M > 10: {gal.loc[gal['lmass'] > 10, 'u_v_err'].median():.3f} mag")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 3. Combining Measurements

### Weighted means

When you have several measurements of the same thing with different uncertainties, the more precise ones should count for more. The **inverse-variance weighted mean** gives each measurement a weight $w_i = 1/\sigma_i^2$:

$$\bar{x} = \frac{\sum w_i x_i}{\sum w_i}, \qquad \sigma_{\bar{x}} = \frac{1}{\sqrt{\sum w_i}}$$

In [ ]:
x = np.array([10.2, 9.8, 10.6])
sigma = np.array([0.2, 0.3, 0.6])

w = 1 / sigma**2
x_mean = np.sum(w * x) / np.sum(w)
x_err = 1 / np.sqrt(np.sum(w))
print(f"Weighted mean: {x_mean:.3f} ± {x_err:.3f}")
print(f"Plain mean:    {np.mean(x):.3f}")

NumPy can do the mean part with `np.average(x, weights=w)`.

### Tension between measurements

A weighted mean only makes sense if the measurements agree. To check, look at how far apart two measurements are compared to their combined uncertainty:

$$T = \frac{|x_1 - x_2|}{\sqrt{\sigma_1^2 + \sigma_2^2}}$$

This is the difference in units of "sigma". Differences of 1 or 2σ happen by chance all the time. Above about 3σ, it's worth asking whether something is wrong. Physicists usually want 5σ before claiming a discovery.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: The Hubble Tension
  </h1>
</div>

Three well-known measurements of the Hubble constant, in km/s/Mpc, are:

| Method | $H_0$ |
|--------|-------|
| Cosmic microwave background, Planck ([2020](https://ui.adsabs.harvard.edu/abs/2020A%26A...641A...6P)) | $67.4 \pm 0.5$ |
| Cepheids and supernovae, SH0ES ([Riess et al. 2022](https://ui.adsabs.harvard.edu/abs/2022ApJ...934L...7R)) | $73.04 \pm 1.04$ |
| Tip of the red giant branch ([Freedman 2021](https://ui.adsabs.harvard.edu/abs/2021ApJ...919...16F)) | $69.8 \pm 1.7$ |

1. Calculate the weighted mean of all three and its uncertainty.
2. Calculate the tension between Planck and SH0ES, and store it in `tension`.
3. Is the weighted mean a sensible summary of what we know about $H_0$? Why or why not?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(tension - 4.88) < 0.05, "tension isn't right yet. Divide the difference by the combined uncertainty."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
h0 = np.array([67.4, 73.04, 69.8])
h0_err = np.array([0.5, 1.04, 1.7])

w = 1 / h0_err**2
print(f"Weighted mean: {np.sum(w * h0) / np.sum(w):.2f} ± {1 / np.sqrt(np.sum(w)):.2f} km/s/Mpc")

tension = abs(h0[0] - h0[1]) / np.sqrt(h0_err[0]**2 + h0_err[1]**2)
print(f"Planck vs SH0ES: {tension:.1f} sigma")

# No: the two most precise measurements disagree by about 5 sigma, so at least one of them has an
# unrecognised systematic error, or our cosmological model is missing something. Averaging them gives
# a precise-looking number that hides the disagreement. This is the famous "Hubble tension".
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 4. Fitting a Model to Data

Fitting means finding the model parameters that best describe your data. The standard approach is **least squares**: choose the parameters that minimise

$$\chi^2 = \sum_i \left(\frac{y_i - f(x_i; \theta)}{\sigma_i}\right)^2$$

where $y_i \pm \sigma_i$ are the data, $f(x; \theta)$ is the model with parameters $\theta$, and each point's contribution is scaled by its uncertainty. Points with small uncertainties pull hardest on the fit.

### A test with fake data

It's always a good idea to test a fitting method on simulated data where you know the right answer. Let's make some straight-line data with known parameters and noise:

In [ ]:
rng = np.random.default_rng(seed=9)

true_slope, true_intercept = 2.0, 1.0
x = np.linspace(0, 10, 20)
y_err = rng.uniform(0.5, 1.5, size=x.size)               # each point has its own uncertainty
y = true_slope * x + true_intercept + rng.normal(0, y_err)

`scipy.optimize.curve_fit` takes a model function, whose first argument is $x$ and whose remaining arguments are the parameters, and returns:

- `popt`: the best-fit parameters
- `pcov`: their **covariance matrix**. The square roots of its diagonal are the parameter uncertainties.

Two arguments are very important:

- `sigma=` gives the uncertainty on each $y$ value. Without it, every point counts equally.
- `absolute_sigma=True` tells `curve_fit` your uncertainties are real, measured values. Without it, `curve_fit` rescales them so the fit looks perfect, and your parameter uncertainties can be badly wrong.

In [ ]:
def line(x, slope, intercept):
    """A straight line."""
    return slope * x + intercept


popt, pcov = curve_fit(line, x, y, sigma=y_err, absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))

print(f"Slope:     {popt[0]:.3f} ± {perr[0]:.3f}   (true value {true_slope})")
print(f"Intercept: {popt[1]:.3f} ± {perr[1]:.3f}   (true value {true_intercept})")

The fit recovers the true values within their uncertainties. With honest uncertainties, the true value should lie within $\pm 1\sigma$ of the fit about 68% of the time, and within $\pm 2\sigma$ about 95% of the time. Try a few different seeds to see this.

### Is the fit any good?

The best fit isn't necessarily a *good* fit. To judge it, look at the $\chi^2$ and the **residuals** (data minus model).

If the model is right and the uncertainties are correct, $\chi^2$ should be about equal to the number of **degrees of freedom**, $\nu = N_\mathrm{data} - N_\mathrm{parameters}$. So the **reduced** $\chi^2_\nu = \chi^2 / \nu$ should be close to 1:

- $\chi^2_\nu \gg 1$: the model doesn't describe the data, or the uncertainties are underestimated.
- $\chi^2_\nu \ll 1$: the uncertainties are probably overestimated, or the points aren't independent.

The **p-value** from the $\chi^2$ distribution gives the probability of a $\chi^2$ at least this large if the model were right.

In [ ]:
model = line(x, *popt)          # *popt unpacks the parameters into the function's arguments
residuals = y - model

chi2 = np.sum((residuals / y_err) ** 2)
dof = len(x) - len(popt)
p_value = stats.chi2.sf(chi2, dof)

print(f"chi2 = {chi2:.1f} for {dof} degrees of freedom, reduced chi2 = {chi2 / dof:.2f}, p = {p_value:.2f}")

Always plot the residuals. A good fit has residuals scattered randomly around zero. A pattern, like a curve, means the model is missing something. `gridspec_kw` lets you make the residual panel shorter than the main one.

In [ ]:
fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(6, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax.errorbar(x, y, yerr=y_err, fmt="o", color="k", ms=4, label="Data")
ax.plot(x, model, color="tab:blue", label=f"Fit: y = {popt[0]:.2f}x + {popt[1]:.2f}")
ax.set_ylabel("y")
ax.legend()

ax_res.errorbar(x, residuals, yerr=y_err, fmt="o", color="k", ms=4)
ax_res.axhline(0, color="tab:blue")
ax_res.set_xlabel("x")
ax_res.set_ylabel("Data - model")

plt.tight_layout()
plt.show()

### Type Ia supernovae and the Hubble constant

Type Ia supernovae are exploding white dwarfs with almost the same peak luminosity, so their apparent brightness tells us their distance. **Pantheon+** ([Scolnic et al. 2022](https://ui.adsabs.harvard.edu/abs/2022ApJ...938..113S)) is a compilation of 1,701 light curves of 1,543 supernovae. Some supernovae were observed by more than one survey, so they appear more than once.

For each one we have the redshift `z` and the **distance modulus** $\mu$ (Part 2), with uncertainty `mu_err`. The distances are calibrated using supernovae in nearby galaxies whose distances are known from Cepheid variable stars. These **calibrators** are flagged with `is_calibrator = 1`, and we'll leave them out of the fits.

In [ ]:
sn = pd.read_csv(get_data("pantheon_plus.csv"))
sn = sn[sn["is_calibrator"] == 0]

print(f"{len(sn)} supernova light curves")
sn.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(sn["z"], sn["mu"], yerr=sn["mu_err"], fmt=".", ms=3, color="k", ecolor="0.75", elinewidth=0.6)
ax.set_xscale("log")
ax.set_xlabel("Redshift")
ax.set_ylabel(r"Distance modulus $\mu$ (mag)")
ax.set_title("The Pantheon+ Hubble diagram")
plt.show()

The distance modulus converts to a distance in Mpc with $d = 10^{(\mu - 25)/5}$.

At low redshift, Hubble's law says $cz = H_0 d$. But below about $z = 0.023$, galaxies' own **peculiar velocities** (a few hundred km/s, from the pull of nearby structures) are a significant fraction of their recession velocity, so we only use supernovae above that. We also stop at $z = 0.1$, where we know Hubble's law starts to break down (Part 2).

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Measuring the Hubble Constant
  </h1>
</div>

1. Select supernovae with $0.023 < z < 0.1$, and calculate their distances `d` in Mpc and velocities `v = c * z` in km/s, with $c = 299{,}792.458$ km/s.
2. The uncertainties are on the distances, so fit the model $d = v / H_0$ to $d$ as a function of $v$. Propagate `mu_err` to an uncertainty on $d$ with $\sigma_d = d \ln(10)\, \sigma_\mu / 5$. Use `curve_fit` with `sigma` and `absolute_sigma=True`, and store the best fit in `H0_fit`.
3. Calculate the reduced $\chi^2$ and comment on it.
4. Plot distance against velocity with your fit, with a residual panel underneath.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert 70 < H0_fit < 72.5, "H0_fit should be about 71 km/s/Mpc. Check your redshift range and that you fitted d against v."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
c = 299792.458

hf = sn[(sn["z"] > 0.023) & (sn["z"] < 0.1)]
d = 10 ** ((hf["mu"] - 25) / 5)
d_err = d * np.log(10) * hf["mu_err"] / 5
v = c * hf["z"]


def hubble_law(v, H0):
    """Distance in Mpc for a recession velocity in km/s."""
    return v / H0


popt, pcov = curve_fit(hubble_law, v, d, p0=[70], sigma=d_err, absolute_sigma=True)
H0_fit, H0_err = popt[0], np.sqrt(pcov[0, 0])

resid = d - hubble_law(v, H0_fit)
chi2 = np.sum((resid / d_err) ** 2)
dof = len(d) - 1
print(f"{len(d)} supernovae: H0 = {H0_fit:.2f} ± {H0_err:.2f} km/s/Mpc, reduced chi2 = {chi2 / dof:.2f}")

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(6.5, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(v, d, yerr=d_err, fmt=".", color="k", ecolor="0.7", ms=4)
v_line = np.linspace(0, v.max() * 1.05, 100)
ax.plot(v_line, hubble_law(v_line, H0_fit), color="tab:blue", label=f"$H_0$ = {H0_fit:.1f} ± {H0_err:.1f} km/s/Mpc")
ax.set_ylabel("Distance (Mpc)")
ax.legend()
ax_res.errorbar(v, resid, yerr=d_err, fmt=".", color="k", ecolor="0.7", ms=4)
ax_res.axhline(0, color="tab:blue")
ax_res.set_xlabel("Velocity cz (km/s)")
ax_res.set_ylabel("Data - model (Mpc)")
plt.tight_layout()
plt.show()

# The reduced chi2 is well below 1. The quoted uncertainties include contributions that are shared
# between supernovae (correlated systematics), so treating them as independent overestimates the scatter.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

You should get $H_0 \approx 71$ km/s/Mpc, with an uncertainty under 1 km/s/Mpc (the statistical part only). But the Pantheon+ distances are calibrated to the SH0ES Cepheids, which give 73. Why the difference?

Look at your residuals: they curve slightly upwards at high velocity. Hubble's law $d = cz/H_0$ is only the first term of a series, and by $z = 0.1$ the expansion history of the Universe already matters. A too-simple model gives a **biased** answer, however small its uncertainty. That's what the next section fixes.

---
## 5. Fitting a Cosmology

`curve_fit` works with any model you can write as a Python function, including one that calls Astropy. Here the model is the distance modulus in a flat universe with Hubble constant $H_0$ and matter density $\Omega_m$ (the rest being dark energy), exactly as in Part 2.

Nonlinear fits need a starting guess, `p0`, and it helps to give `bounds` that keep the parameters physical.

In [ ]:
def distance_modulus(z, H0, Om0):
    """Distance modulus in a flat Lambda-CDM universe."""
    return FlatLambdaCDM(H0=H0, Om0=Om0).distmod(z).value


fit_sn = sn[sn["z"] > 0.01]

popt, pcov = curve_fit(distance_modulus, fit_sn["z"], fit_sn["mu"], p0=[70, 0.3],
                       sigma=fit_sn["mu_err"], absolute_sigma=True, bounds=([50, 0], [100, 1]))
perr = np.sqrt(np.diag(pcov))

chi2_lcdm = np.sum(((fit_sn["mu"] - distance_modulus(fit_sn["z"], *popt)) / fit_sn["mu_err"]) ** 2)
dof = len(fit_sn) - 2

print(f"H0  = {popt[0]:.2f} ± {perr[0]:.2f} km/s/Mpc")
print(f"Om0 = {popt[1]:.3f} ± {perr[1]:.3f}")
print(f"chi2 = {chi2_lcdm:.0f} for {dof} degrees of freedom")

Using the full model, $H_0$ comes out at about 73, consistent with the SH0ES calibration. We also measure $\Omega_m \approx 0.35$, so about two thirds of the Universe's energy is in **dark energy**. The published Pantheon+ result is $\Omega_m = 0.334 \pm 0.018$. Ours differs slightly because we've ignored the correlations between supernovae, which the real analysis includes with a full **covariance matrix**.

The covariance matrix also tells us how the parameter uncertainties are linked. The **correlation coefficient** between two parameters is their covariance divided by the product of their uncertainties:

In [ ]:
corr = pcov[0, 1] / (perr[0] * perr[1])
print(f"Correlation between H0 and Om0: {corr:.2f}")

A negative correlation means a slightly higher $H_0$ can be partly compensated by a slightly lower $\Omega_m$. When parameters are strongly correlated, quoting their uncertainties separately hides this, which is why papers show contour plots of the joint constraints.

### Comparing models

Is there really evidence for dark energy? We can compare our best fit with a universe made only of matter ($\Omega_m = 1$, no dark energy), the standard picture before 1998. The model with the lower $\chi^2$ fits better, and the size of the difference, $\Delta\chi^2$, says by how much. For one extra parameter, a $\Delta\chi^2$ of 9 corresponds to roughly 3σ.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Evidence for Dark Energy
  </h1>
</div>

1. Fit a matter-only model to `fit_sn`, with $\Omega_m$ fixed at 1 and only $H_0$ free. *Hint: write a new function with only `z` and `H0` as arguments, which calls `distance_modulus(z, H0, 1.0)`.*
2. Calculate its $\chi^2$ and the difference `delta_chi2` from `chi2_lcdm`. How strong is the preference for dark energy?
3. Plot the residuals of both models (data minus model) against redshift on the same axes, with a log x axis. Where do the two models differ most?
4. **Extension:** Refit the Lambda-CDM model using only supernovae with $z < 0.1$. What happens to the uncertainty on $\Omega_m$, and why?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert delta_chi2 > 500, "delta_chi2 should be several hundred. Did you fix Om0 = 1 in the matter-only model?"
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
def matter_only(z, H0):
    """Distance modulus in a flat universe with no dark energy."""
    return distance_modulus(z, H0, 1.0)


popt_m, pcov_m = curve_fit(matter_only, fit_sn["z"], fit_sn["mu"], p0=[70], sigma=fit_sn["mu_err"], absolute_sigma=True)
chi2_matter = np.sum(((fit_sn["mu"] - matter_only(fit_sn["z"], *popt_m)) / fit_sn["mu_err"]) ** 2)
delta_chi2 = chi2_matter - chi2_lcdm
print(f"Matter only: H0 = {popt_m[0]:.2f}, chi2 = {chi2_matter:.0f}, delta chi2 = {delta_chi2:.0f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(fit_sn["z"], fit_sn["mu"] - distance_modulus(fit_sn["z"], *popt), s=4, color="tab:blue", alpha=0.5, label=r"$\Lambda$CDM")
ax.scatter(fit_sn["z"], fit_sn["mu"] - matter_only(fit_sn["z"], *popt_m), s=4, color="tab:orange", alpha=0.5, label="Matter only")
ax.axhline(0, color="k", lw=1)
ax.set_xscale("log")
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel("Redshift")
ax.set_ylabel(r"$\mu$ - model (mag)")
ax.legend()
plt.show()

low = fit_sn[fit_sn["z"] < 0.1]
popt_low, pcov_low = curve_fit(distance_modulus, low["z"], low["mu"], p0=[70, 0.3], sigma=low["mu_err"],
                               absolute_sigma=True, bounds=([50, 0], [100, 1]))
print(f"z < 0.1 only: Om0 = {popt_low[1]:.2f} ± {np.sqrt(pcov_low[1, 1]):.2f}")

# Delta chi2 of several hundred is overwhelming evidence. The matter-only model fails at z > 0.3, where
# distant supernovae are fainter (further away) than it predicts: the expansion has been accelerating.
# Using only z < 0.1 the uncertainty on Om0 is several times larger, because at low redshift every
# cosmology looks like Hubble's law. You need distant objects to measure the expansion history.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

This is a version of the measurement that won the 2011 Nobel Prize in Physics: distant supernovae are fainter than a decelerating universe allows, so the expansion is accelerating.

**A note on what we've skipped.** `curve_fit` assumes the uncertainties are Gaussian and independent, and it describes the parameter uncertainties with a single covariance matrix. Real analyses often need more: full covariance matrices between data points, priors, and parameter distributions that aren't Gaussian. The standard tool for this is **Markov chain Monte Carlo** (MCMC), for example with the [emcee](https://emcee.readthedocs.io/) package, which you'll likely meet in your project.

---
## 6. Comparing Samples and the Bootstrap

### Are two distributions different?

The **Kolmogorov-Smirnov (KS) test** asks whether two samples could have come from the same distribution. It compares their cumulative distributions and returns the largest gap between them (the statistic) and a p-value. Let's compare the stellar masses of quiescent and star-forming galaxies:

In [ ]:
sample = gal[(gal["z_best"] > 0.5) & (gal["z_best"] < 2.5) & (gal["lmass"] > 9)]
q_mass = sample.loc[sample["quiescent"], "lmass"]
sf_mass = sample.loc[~sample["quiescent"], "lmass"]

result = stats.ks_2samp(q_mass, sf_mass)
print(f"KS statistic = {result.statistic:.2f}, p-value = {result.pvalue:.1e}")
print(f"Median log mass: quiescent {q_mass.median():.2f}, star-forming {sf_mass.median():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for masses, label, colour in [(sf_mass, "Star-forming", "tab:blue"), (q_mass, "Quiescent", "tab:red")]:
    ax.plot(np.sort(masses), np.arange(1, len(masses) + 1) / len(masses), color=colour, lw=2, label=label)
ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
ax.set_ylabel("Cumulative fraction")
ax.legend()
plt.show()

The p-value is tiny, so the two mass distributions are clearly different: quiescent galaxies are more massive.

**Be careful with p-values for large samples.** With thousands of objects, even a tiny, unimportant difference gives a very small p-value. A p-value tells you whether a difference is likely to be real, not whether it's big enough to matter. Always report the size of the difference too, like the difference in medians here.

### Uncertainty on a fraction

In Part 3 you found that about half of the most massive galaxies are quiescent. How certain is that? For $k$ successes out of $N$, the fraction $p = k/N$ has a **binomial** uncertainty

$$\sigma_p = \sqrt{\frac{p(1 - p)}{N}}$$

In [ ]:
massive = sample[sample["lmass"] > 11]
p = massive["quiescent"].mean()
n = len(massive)
print(f"Quiescent fraction above 10^11 Msun: {p:.2f} ± {np.sqrt(p * (1 - p) / n):.2f} ({n} galaxies)")

### The bootstrap

What about quantities with no simple formula, like the uncertainty on a median? The **bootstrap** estimates it from the data itself. The idea is to make many new samples by drawing from your data **with replacement**, so some objects get picked more than once and some not at all, then recalculate the quantity for each. The spread of the results estimates its uncertainty.

In [ ]:
rng = np.random.default_rng(seed=7)
masses = q_mass.to_numpy()

boot_medians = []
for i in range(5000):
    resample = rng.choice(masses, size=len(masses), replace=True)
    boot_medians.append(np.median(resample))

print(f"Median log mass of quiescent galaxies: {np.median(masses):.3f} ± {np.std(boot_medians):.3f}")

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Bootstrapping a Fraction
  </h1>
</div>

1. Use the bootstrap, with 10,000 resamples, to estimate the uncertainty on the quiescent fraction of galaxies in `massive`. Store it in `boot_err`. How does it compare with the binomial formula?
2. Calculate the quiescent fraction and its bootstrap uncertainty in each of the mass bins `[9, 9.5, 10, 10.5, 11, 12]` from Part 3, and plot them with error bars against the centre of each bin.
3. Are the differences between neighbouring bins significant?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(boot_err - 0.066) < 0.008, "boot_err should be close to the binomial uncertainty of about 0.066."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
rng = np.random.default_rng(seed=11)
is_q = massive["quiescent"].to_numpy()
boot = [rng.choice(is_q, size=len(is_q), replace=True).mean() for i in range(10_000)]
boot_err = np.std(boot)
print(f"Bootstrap: {is_q.mean():.3f} ± {boot_err:.3f}, binomial: ± {np.sqrt(p * (1 - p) / n):.3f}")

edges = [9, 9.5, 10, 10.5, 11, 12]
centres, fractions, errors = [], [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    q_bin = sample.loc[(sample["lmass"] >= lo) & (sample["lmass"] < hi), "quiescent"].to_numpy()
    boot = [rng.choice(q_bin, size=len(q_bin), replace=True).mean() for i in range(2000)]
    centres.append((lo + hi) / 2)
    fractions.append(q_bin.mean())
    errors.append(np.std(boot))
    print(f"{lo}-{hi}: {q_bin.mean():.3f} ± {np.std(boot):.3f} ({len(q_bin)} galaxies)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(centres, fractions, yerr=errors, fmt="o-", color="tab:red", capsize=3)
ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
ax.set_ylabel("Quiescent fraction")
ax.set_ylim(0, 0.8)
plt.show()

# Every step up in mass is many times larger than the uncertainties, except perhaps the last, where
# the small number of very massive galaxies makes the uncertainty larger.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

<div class="practical-end" style="
  background: #f5f5f5;
  border-radius: 12px;
  padding: 20px 24px;
  text-align: center;
  border: 1px solid #ddd;
  margin: 2rem 0;
">
  <p style="font-size: 1.5rem; margin: 0 0 8px; text-align: center;">🎉</p>
  <h1 style="margin: 0 0 4px; color: #111; text-align: center;">End of Part 5</h1>
  <p style="margin: 0; color: #555; font-size: 0.9rem; text-align: center;">
    Well done! You've finished Part 5.
  </p>
</div>

<style>
@media (prefers-color-scheme: dark) {
  .practical-end { background: #1e1e1e !important; border-color: #333 !important; }
  .practical-end h1 { color: #f0f0f0 !important; }
  .practical-end p  { color: #999     !important; }
}
</style>

## What's Next?

In **Part 6** you'll put everything together in a few short, self-contained galaxy evolution examples, and find out how to keep going with your own project.

<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/06-Putting-It-Together.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Want to learn more?

- Hogg, Bovy & Lang (2010), [*Data analysis recipes: Fitting a model to data*](https://arxiv.org/abs/1008.4686), a short, readable guide to doing fitting properly
- Ivezić et al., [*Statistics, Data Mining, and Machine Learning in Astronomy*](https://www.astroml.org/), the standard textbook, with Python code for every figure
- Andrae et al. (2010), [*Dos and don'ts of reduced chi-squared*](https://arxiv.org/abs/1012.3754)
- [emcee documentation](https://emcee.readthedocs.io/), for MCMC fitting
- [SciPy statistics reference](https://docs.scipy.org/doc/scipy/reference/stats.html)